In [1]:
!pip install -q chromadb sentence-transformers --no-warn-conflicts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00


In [2]:
import json
import os
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize Chroma client with a persistent local path
checkpoint_path = "/kaggle/working/chroma_legal_db"
chroma_client = chromadb.PersistentClient(path=checkpoint_path)

# Create or fetch a collection with Distance function 'cosine'
collection = chroma_client.get_or_create_collection(
    name="indian_corporate_laws", 
    metadata={"hnsw:space": "cosine"}
)

In [3]:
import numpy as np
import pandas as pd
import kagglehub

# Input files path
data_dir = '/kaggle/input/datasets/vinayak9421/corporate-laws-of-india-structured-json'
print('Files available: ', data_dir)
for dirname, _, filenames in os.walk(data_dir):
    for filename in filenames:
        print(filename)

Files available:  /kaggle/input/datasets/vinayak9421/corporate-laws-of-india-structured-json
Payment of wages 1937.json
Sexual Harassment of Women At Workplace Act2013.json
The Code on Wages 2019.json
The Payment Of Gratuity Act 1972.json
The Insolvency And Bankruptcy Code 2016.json
The Minimum Wages Act 1948.json
the_industrial_disputes_act_1947.json
A2013-14.json
llp act 2008.json
The Apprentices Act1961.json
Maternity Benefit Act 1961.json
The Inter-State Migrant Workmen Regulation Of Employment And Conditions  Of Service Act 1979.json
The Industrial Employment.json
wages 2017.json


In [4]:
documents_to_add = []
metadatas_to_add = []
ids_to_add = []
unknown_counter = 0

# Loop through all JSON files in the dataset
for file_name in os.listdir(data_dir):
    if file_name.endswith('.json'):
        file_path = os.path.join(data_dir, file_name)
        
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                act_data = json.load(f)
            except Exception as e:
                print(f"Skipping malformed file {file_name}: {e}")
                continue
                
            act_name = act_data.get("act_name", file_name)
            
            # Loop through sections inside the Act
            for section in act_data.get("sections", []):
                sec_num = section.get("number")
                text_content = section.get("text", "").strip()
                keywords = ", ".join(section.get("keywords", []))
                
                if text_content:
                    # Resolve ID string naming collision safety
                    if sec_num is not None and str(sec_num).strip() != "":
                        clean_sec_num = str(sec_num).strip()
                        doc_id = f"{file_name}_sec_{clean_sec_num}"
                    else:
                        unknown_counter += 1
                        doc_id = f"{file_name}_sec_unknown_{unknown_counter}"
                    
                    documents_to_add.append(text_content)
                    metadatas_to_add.append({
                        "act_name": act_name,
                        "section_number": str(sec_num) if sec_num else "Unknown Section",
                        "keywords": keywords
                    })
                    ids_to_add.append(doc_id)

print(f"Successfully processed {len(documents_to_add)} unique documents!")

Successfully processed 761 unique documents!


In [5]:
print("Sample of data loaded.......... ")
print("=" * 80)
print(documents_to_add[0])
print("-" * 50)
print(metadatas_to_add[0])
print("-" * 50)
print(ids_to_add[0])

Sample of data loaded.......... 
section 15. (2) The Authority may refuse to entertain an applic ation which is 
insufficiently stamped or otherwise incomplete and,  if he so refuses, shall return it 
at once with an indication of the defects.  If the application is presented again 
after the defects have been made good, the date of representation shall be deemed 
to be the date of presentation for the purpose of t he proviso sub-section (2) of
--------------------------------------------------
{'act_name': 'Payment of wages 1937.json', 'section_number': 'Unknown Section', 'keywords': 'application, stamped, incomplete, defects, return, representation, presentation date'}
--------------------------------------------------
Payment of wages 1937.json_sec_unknown_1


In [6]:
print("Loading data into Vector DB - ")
# Load a strong open-weights embedding model
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

print("Generating vector embedding matrix arrays...")
embeddings = model.encode(documents_to_add, show_progress_bar=True).tolist()

# Safe upsert execution
collection.add(
    embeddings=embeddings,
    documents=documents_to_add,
    metadatas=metadatas_to_add,
    ids=ids_to_add
)
print("Vector Database safely written and deployed without collisions!")


Loading data into Vector DB - 


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating vector embedding matrix arrays...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Vector Database safely written and deployed without collisions!


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Install the cross-encoder library
!pip install -q sentence-transformers --no-warn-conflicts

from sentence_transformers import CrossEncoder

# Load a highly efficient, small cross-encoder model
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# --- SECTION 1: LLM INFERENCE GATEWAY ---
# (Assumes your model and tokenizer cells are already loaded in memory)
# If they are not, uncomment the two lines below:
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
# llm_model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", torch_dtype=torch.float16, device_map="auto")


# --- SECTION 2: UPGRADED RERANKING RETRIEVER ---
def search_legal_db(user_query, target_act=None, top_k=2):
    """
    Stage 1: Fetches a wide pool of vectors from ChromaDB.
    Stage 2: Rerank matching contexts using Cross-Encoder to bubble up the best results.
    """
    query_vector = model.encode([user_query]).tolist()
    where_filter = {"act_name": target_act} if target_act else None
    
    # Grab top 10 from vector space
    raw_results = collection.query(
        query_embeddings=query_vector,
        n_results=10, 
        where=where_filter
    )
    
    documents = raw_results['documents'][0]  # Unpacking ChromaDB structural nesting
    metadatas = raw_results['metadatas'][0]
    
    if not documents:
        return ""

    # Cross-Encoder matching calculation
    pairs = [[user_query, doc] for doc in documents]
    scores = reranker_model.predict(pairs)
    
    # Sort descending based on deep semantic match
    reranked_data = sorted(
        zip(scores, documents, metadatas), 
        key=lambda x: x[0], 
        reverse=True
    )
    
    # Construct clean string context block for the LLM
    formatted_context_for_llm = ""
    for score, doc, meta in reranked_data[:top_k]:
        formatted_context_for_llm += f"Source: {meta['act_name']} (Section {meta['section_number']})\nLegal Text: {doc}\n\n"
        
    return formatted_context_for_llm

search_legal_db("Find the Wages law")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

'Source: the_industrial_disputes_act_1947.json (Section Unknown Section)\nLegal Text: “wages ” means all remuneration capable of being expressed in terms of money, which would, if the terms of employment, expressed or implied, were fulfilled, be payable to a workman in respect of his employment or of work done in such employment, and includes—\n\nSource: wages 2017.json (Section Unknown Section)\nLegal Text: For section 6 of the Payment of W ages Act, 1936, the following section shall be substituted, namely:—\n"6. All wages shall be paid in current coin or currency notes or by cheque or by crediting the wages in the bank account of the employee: Provided that the appropriate Government may, by notification in the Official Gazette, specify the industrial or other establishment, the employer of which shallpay to every person employed in such industrial or other establishment, the wagesonly by cheque or by crediting the wages in his bank account."\n\n'

In [8]:
!pip install -q transformers accelerate --no-warn-conflicts
!pip install -q transformers accelerate bitsandbytes --no-warn-conflicts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 47.7 MB/s eta 0:00:00


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

# 1. Authenticate globally across the session background
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("KF_TOKEN")
login(token=hf_token)
os.environ["KF_TOKEN"] = hf_token

model_id = "google/gemma-2-2b-it"

print(f"🚀 Downloading and loading {model_id} from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Load model directly using 16-bit float (perfectly fits the T4 GPU)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ Success! Gemma 2B text model is active and online.")


🚀 Downloading and loading google/gemma-2-2b-it from Hugging Face...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

✅ Success! Gemma 2B text model is active and online.


In [10]:
def ask_indian_law_rag(user_question):
    """
    Takes user question, triggers the advanced retriever, constructs prompt layers,
    and runs deterministic generation through Gemma-2B.
    """

    # 1. Dynamically check if a GPU is active, otherwise fall back to CPU safely
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running inference execution loop on target device: {device.upper()}")
        
    # 1. Fetch optimized context from the reranker function above
    retrieved_context = search_legal_db(user_question, top_k=2)
    
    if not retrieved_context.strip():
        return "I could not retrieve any relevant statutory context from the legal database."

    # 2. Format message array utilizing official Gemma structural tokens
    messages = [
        {
            "role": "user",
            "content": (
                "You are an expert AI Legal Assistant specializing in Indian Labour and Corporate Law. "
                "Answer the user's question using ONLY the provided legal text sections. "
                "If the answer cannot be confidently deduced from the text, explicitly state: "
                "'I do not find the answer in the provided context.' Do not fabricate laws. "
                "You must explicitly cite the [Act Name] and [Section Number] when giving your answer.\n\n"
                f"RETIREVED LEGAL CONTEXT:\n{retrieved_context}\n\n"
                f"USER QUESTION: {user_question}"
            )
        }
    ]

    # 3. Apply chat tokenizer boundaries
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # 4. Generate highly constrained answers (low temperature avoids legal hallucinations)
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs, 
            max_new_tokens=350, 
            temperature=0.1, 
            do_sample=False
        )
    
    # 5. Decode just the newly generated reply sequence strings
    input_length = inputs.input_ids.shape[-1]
    generated_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return answer.strip()

print("🏁 System Synced! 'ask_indian_law_rag' is defined and connected to Cross-Encoding.")

🏁 System Synced! 'ask_indian_law_rag' is defined and connected to Cross-Encoding.


In [11]:
def advanced_hybrid_retriever(user_question, top_n_to_llm=2):
    # 1. Grab a wide net of documents from ChromaDB (Stage 1)
    query_vector = model.encode([user_question]).tolist()
    raw_results = collection.query(query_embeddings=query_vector, n_results=10)
    
    documents = raw_results['documents'][0]
    metadatas = raw_results['metadatas'][0]
    
    # 2. Pair the question with each document for the reranker
    pairs = [[user_question, doc] for doc in documents]
    scores = reranker_model.predict(pairs)
    
    # 3. Sort documents by their new relevancy scores
    reranked_results = sorted(zip(scores, documents, metadatas), key=lambda x: x[0], reverse=True)
    
    # 4. Format the top N results for the LLM
    final_context = ""
    for score, doc, meta in reranked_results[:top_n_to_llm]:
        final_context += f"Source: {meta['act_name']} (Section {meta['section_number']})\nLegal Text: {doc}\n\n"
        
    return final_context


In [12]:
# Install Gradio
!pip install -q gradio --no-warn-conflicts

import gradio as gr

# Wrap your RAG function into Gradio's expected format
def chatbot_ui_fn(message, history):
    try:
        # Call your end-to-end RAG function
        bot_response = ask_indian_law_rag(message)
        return bot_response
    except Exception as e:
        return f"⚠️ Error: {str(e)}"

# Create a clean, professional legal chat interface
demo = gr.ChatInterface(
    fn=chatbot_ui_fn,
    title="⚖️ Indian Corporate & Labour Law AI Assistant",
    description="Ask any legal questions regarding Indian Corporate Acts, Payment of Wages, or Workplace Safety. The AI will answer using verified statutory text and provide citations.",
    examples=[
        "What is the penalty if an employer fails to pay wages on time?",
        "What is the minimum age required for an apprentice?",
        "What constitutes sexual harassment at a workplace?"
    ],
    theme="soft"
)

# Launch it! (inline=True makes it display directly inside your notebook cell)
demo.launch(inline=True, share=True)


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://49e024e37cb207b5b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
